# Prompt 버전 관리와 A/B 테스트

프롬프트가 여러 번 수정되면서 의도가 흐려지는 **Prompt Drift** 현상을 방지하기 위해,
프롬프트를 코드처럼 **SemVer**로 버전 관리하고 **A/B 테스트**로 검증하는 방법을 실습한다.

### 학습 목표

1. 프롬프트를 딕셔너리 기반 Registry로 관리하는 패턴을 익힌다
2. LM Judge로 프롬프트 버전 간 성능을 정량 비교한다
3. 해시 기반 A/B 테스트로 트래픽을 분배하는 로직을 구현한다
4. Langfuse에서 버전별 트레이스를 필터링하고 비교한다

---

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler
import hashlib

## 1. Prompt Registry — 프롬프트를 코드처럼 관리하기

프롬프트도 코드다. 변경할 때마다 **SemVer** 규칙으로 버전을 매기고, 변경 이유를 기록한다.

| 변경 유형 | 버전 증가 | 예시 |
|-----------|----------|------|
| 형식만 수정 | Patch (v1.0.0→v1.0.1) | 줄바꿈 조정 |
| 동작 개선 | Minor (v1.0.0→v1.1.0) | 예시 추가 |
| 역할·목적 변경 | Major (v1.0.0→v2.0.0) | 시스템 프롬프트 전면 교체 |

In [ ]:
# ── Prompt Registry 구현 ──
PROMPT_REGISTRY: dict[str, dict] = {}


def register_prompt(name: str, version: str, content: str, changelog: str = ""):
    """프롬프트를 레지스트리에 등록한다."""
    key = f"{name}@{version}"
    PROMPT_REGISTRY[key] = {
        "name": name,
        "version": version,
        "content": content,
        "changelog": changelog,
    }
    print(f"✅ 등록: {key}")


def get_prompt(name: str, version: str) -> str:
    """특정 버전의 프롬프트를 가져온다."""
    return PROMPT_REGISTRY[f"{name}@{version}"]["content"]


def list_versions(name: str):
    """프롬프트의 모든 버전을 출력한다."""
    versions = [v for v in PROMPT_REGISTRY.values() if v["name"] == name]
    for v in sorted(versions, key=lambda x: x["version"]):
        print(f"  {v['version']}: {v['changelog']}")

In [ ]:
# ── 고객 상담 프롬프트 3개 버전 등록 ──

register_prompt(
    name="customer-support",
    version="v1.0.0",
    content="당신은 고객 상담원입니다. 고객의 질문에 답변하세요.",
    changelog="초기 버전 — 최소한의 지시만 포함",
)

register_prompt(
    name="customer-support",
    version="v1.1.0",
    content="""당신은 전자상거래 플랫폼의 고객 상담원입니다.

규칙:
- 항상 공손하고 전문적인 어조로 답변합니다.
- 불확실한 내용은 추측하지 않고 "확인 후 안내드리겠습니다"라고 답합니다.
- 답변 마지막에 추가 도움이 필요한지 묻습니다.""",
    changelog="톤 가이드 + 불확실성 처리 규칙 추가",
)

register_prompt(
    name="customer-support",
    version="v2.0.0",
    content="""당신은 **ShopEasy** 전자상거래 플랫폼의 시니어 고객 상담원입니다.

## 답변 형식
1. 고객 문의를 한 문장으로 요약
2. 구체적 해결 방안을 단계별로 안내
3. 추가 문의 여부 확인

## 정책
- 환불: 구매 후 14일 이내, 미개봉 상품만 가능
- 교환: 구매 후 30일 이내, 동일 가격대 상품
- 배송: 서울 1~2일, 지방 2~3일

## 주의사항
- 정책 범위를 벗어나는 요청 → "담당 부서에 확인 후 안내드리겠습니다"
- 개인정보(카드번호, 비밀번호)를 절대 요청하지 않음""",
    changelog="전면 개편 — 구조화된 형식 + 구체적 정책 포함",
)

print("\n📋 등록된 버전:")
list_versions("customer-support")

## 2. 버전별 응답 비교

같은 질문에 대해 각 버전이 어떻게 다르게 응답하는지 확인한다.
Langfuse에 `langfuse_tags`로 버전을 태깅하여 나중에 필터링할 수 있게 한다.

In [ ]:
test_questions = [
    "배송이 너무 늦어요. 서울인데 주문한 지 5일 됐어요.",
    "이 상품 환불하고 싶은데 3주 전에 샀어요.",
    "제 카드번호 알려드릴 테니 직접 결제 처리해주세요.",
]

versions = ["v1.0.0", "v1.1.0", "v2.0.0"]
results = {}  # {version: [{question, answer}, ...]}

for version in versions:
    system_prompt = get_prompt("customer-support", version)
    llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
    handler = CallbackHandler()

    results[version] = []
    for q in test_questions:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": q},
        ]
        response = llm.invoke(
            messages,
            config={
                "callbacks": [handler],
                "metadata": {
                    "langfuse_tags": [f"prompt-{version}", "versioning-lab"],
                    "langfuse_session_id": "prompt-versioning-lab",
                    "prompt_version": version,
                },
            },
        )
        results[version].append({"question": q, "answer": response.content})
        print(f"[{version}] Q: {q[:30]}... ✓")

In [ ]:
# ── 응답 비교 ──
for q_idx, q in enumerate(test_questions):
    print(f"\n{'=' * 70}")
    print(f"Q: {q}")
    for version in versions:
        answer = results[version][q_idx]["answer"]
        preview = answer[:200] + "..." if len(answer) > 200 else answer
        print(f"\n--- {version} ---")
        print(preview)

## 3. LM Judge로 정량 비교

사람이 모든 응답을 일일이 평가할 수 없다.
**LLM을 평가자(Judge)로 활용**하여 두 버전의 응답을 비교한다.

In [ ]:
def pairwise_judge(question: str, answer_a: str, answer_b: str, label_a: str, label_b: str) -> dict:
    """두 응답을 비교하여 승자를 판정한다."""
    judge = ChatOpenAI(model="gpt-5.4", temperature=0)
    handler = CallbackHandler()

    prompt = f"""두 고객 상담 응답을 비교 평가하세요.

[질문]
{question}

[A 응답]
{answer_a}

[B 응답]
{answer_b}

평가 기준:
1. 정확성 — 정책에 맞는 답변인가
2. 전문성 — 어조가 적절한가
3. 완결성 — 필요한 정보를 모두 제공하는가

반드시 아래 형식으로만 답하세요:
승자: A 또는 B 또는 TIE
이유: (한 문장)"""

    response = judge.invoke(
        prompt,
        config={
            "callbacks": [handler],
            "metadata": {
                "langfuse_tags": ["lm-judge", "versioning-lab"],
                "comparison": f"{label_a}_vs_{label_b}",
            },
        },
    )
    return {"comparison": f"{label_a} vs {label_b}", "result": response.content}

In [ ]:
# ── v1.0.0 vs v2.0.0 Pairwise 비교 ──
print("📊 LM Judge 평가: v1.0.0 vs v2.0.0\n")

for q_idx, q in enumerate(test_questions):
    a = results["v1.0.0"][q_idx]["answer"]
    b = results["v2.0.0"][q_idx]["answer"]
    verdict = pairwise_judge(q, a, b, "v1.0.0", "v2.0.0")
    print(f"Q: {q[:50]}...")
    print(f"   {verdict['result']}\n")

## 4. A/B 테스트 트래픽 분배

프롬프트를 바꾸기 전에 A/B 테스트로 검증한다.
사용자 ID의 해시값으로 트래픽을 균등 분배하면, **같은 사용자는 항상 같은 버전**을 받는다.

In [ ]:
def get_prompt_version_ab(user_id: str, versions: list[str] = ["v1.1.0", "v2.0.0"]) -> str:
    """user_id의 해시값으로 버전을 결정한다. 같은 유저는 항상 같은 버전을 받는다."""
    hash_val = int(hashlib.md5(user_id.encode()).hexdigest(), 16)
    return versions[hash_val % len(versions)]


# ── 1000명 시뮬레이션 ──
from collections import Counter

assignments = Counter()
for i in range(1000):
    version = get_prompt_version_ab(f"user-{i}")
    assignments[version] += 1

print("🎯 A/B 트래픽 분배 결과 (1000명 시뮬레이션):")
for version, count in sorted(assignments.items()):
    print(f"  {version}: {count}명 ({count / 10:.1f}%)")

In [ ]:
# ── A/B 테스트 실제 실행 ──
# 5명의 가상 유저가 같은 질문을 했을 때, 버전별 응답 비교

sample_users = ["user-42", "user-77", "user-128", "user-256", "user-999"]
question = "환불 기한이 지났는데 환불 가능한가요?"
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
handler = CallbackHandler()

print(f"Q: {question}\n")
for user_id in sample_users:
    version = get_prompt_version_ab(user_id)
    system_prompt = get_prompt("customer-support", version)

    response = llm.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        config={
            "callbacks": [handler],
            "metadata": {
                "langfuse_tags": [f"ab-test", f"prompt-{version}"],
                "langfuse_user_id": user_id,
                "langfuse_session_id": "ab-test-lab",
                "prompt_version": version,
            },
        },
    )
    preview = response.content[:120] + "..." if len(response.content) > 120 else response.content
    print(f"  [{user_id}] → {version}: {preview}")

## 5. Langfuse에서 확인할 포인트

Langfuse UI(`http://localhost:3000`)에서 다음을 확인해보세요:

1. **Traces → Tags 필터**: `prompt-v1.0.0`, `prompt-v2.0.0` 등으로 필터링하여 버전별 트레이스 비교
2. **Sessions**: `prompt-versioning-lab`, `ab-test-lab` 세션으로 그룹화된 트레이스 확인
3. **Metadata**: 각 트레이스의 `prompt_version` 메타데이터 확인
4. **LM Judge 트레이스**: `lm-judge` 태그로 필터링하여 평가 과정 확인
5. **Latency/Cost**: 프롬프트 길이에 따른 응답 시간과 비용 차이 비교

---

## 🔧 실습: 프롬프트 v3.0.0 작성 및 테스트

**목표**: v2.0.0의 약점을 보완하는 v3.0.0 프롬프트를 작성하고 LM Judge로 비교한다.

**가이드**:
- v2.0.0 응답에서 개선할 점을 찾는다 (예: 공감 표현 부족, 대안 제시 미흡)
- v3.0.0에 개선 사항을 반영한다
- 동일한 테스트 질문으로 평가한다

In [ ]:
# TODO: v3.0.0 프롬프트를 작성하세요
register_prompt(
    name="customer-support",
    version="v3.0.0",
    content="""여기에 개선된 프롬프트를 작성하세요.

v2.0.0 대비 개선 포인트:
- ???
""",
    changelog="TODO: 변경 이유를 작성하세요",
)

# TODO: 테스트 실행
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
handler = CallbackHandler()

for q in test_questions:
    system_prompt = get_prompt("customer-support", "v3.0.0")
    response = llm.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": q},
        ],
        config={
            "callbacks": [handler],
            "metadata": {
                "langfuse_tags": ["prompt-v3.0.0", "versioning-lab"],
                "prompt_version": "v3.0.0",
            },
        },
    )
    print(f"Q: {q[:40]}...")
    print(f"A: {response.content[:200]}\n")

In [ ]:
# TODO: v2.0.0 vs v3.0.0 LM Judge 비교를 실행하세요
# pairwise_judge 함수를 활용하세요